In [0]:
%skip
# Not able to access this data directly by giving path, thats where concept of external location comes in
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net

In [0]:
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices

In [0]:
%skip
%sql
drop table deltacatalog.deltadb.invoices

In [0]:
%skip
spark.sparkContext.setJobGroup("00 DML Operations on Delta Tables","CREATING CATALOG AND SCHEMA")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS deltacatalog;
CREATE SCHEMA IF NOT EXISTS deltacatalog.deltadb;

In [0]:
%skip
spark.sparkContext.setJobGroup("00 DML Operations on Delta Tables","Reading Data from Parquet file")

In [0]:
%sql
SELECT * FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet` LIMIT 5

In [0]:
%skip
spark.sparkContext.setJobGroup("00 DML Operations on Delta Tables","Creating delta table using parquet location")

In [0]:
%sql
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices 
AS SELECT * FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`;

In [0]:
%skip
spark.sparkContext.setJobGroup("00 DML Operations on Delta Tables","Describe history on Delta Table")

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices;

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices
select * from PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices;

In [0]:
%sql
select Min(customer_id),Max(customer_id), count(*)  from deltacatalog.deltadb.invoices

In [0]:
%sql
select * from deltacatalog.deltadb.invoices
where customer_id=1;

In [0]:
%sql
Update deltacatalog.deltadb.invoices
SET quantity=10
where customer_id=1;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices;

In [0]:
%sql
DELETE from deltacatalog.deltadb.invoices
where customer_id=99;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices;

In [0]:
%sql
insert into deltacatalog.deltadb.invoices
select * from parquet.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_201_99457.parquet`;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices;

- To **Summarize** concept behind delta lake is the use of a transaction layer on top of your data.
- So data is in form of parquet file and there is a transaction layer called '_delta_log'.
- Whenever a transaction is performed all of data is written inside a JSON file.
- This transaction in a data lake is an atomic unit of work.
- It groups one or more operations together, it can be insert, update, delete, These transactions either complete successfully or fail altogether.

**How does DeltaLake Compute the Latest State?**
- Just do a summation.
- Some parquets were added, some were removed, some where removed and dv was also removed and new one was added with new deletion vector. So just do summation of what finally stays.
_ Thats what JSON files help delta do.

**How does Delta Scale ?**
- Delta creates JSON File for every transaction.
- In companies there are lot of tables and each table is made out of lot of transactions, so there is going to be number of JSON files and which will consume lot of compute power to read JSON.
- So how does delta scale? How does it handle reading massive amount of JSON file.
- Suppose I have done 50 inserts to table. what will happen?

In [0]:
for i in range(50):
    spark.sql("""
              INSERT INTO deltacatalog.deltadb.invoices
            select * from PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
              """)
    print(f"{i}th INSERT COMPLETED")

- 50 Inserts completed , how does deltaLake looks like now?
- Lot of parquet files are added in unique uuid of table folder.
- In _delta_log we see JSON files named compacted jsons ex: 00000000000000000001.00000000000000000006.compacted.json
- That means compacted JSON is created for every 6 JSON files. 1-6 all of info is dumped into 1.
- It is created so that spark doesnt have to go through the overhead of opening and closing those files.
- So we have :
   - 00000000000000000001.00000000000000000006.compacted.json : 1-6
   - 00000000000000000007.00000000000000000012.compacted.json : 7-12
   - 00000000000000000013.00000000000000000018.compacted.json : 13-18
   - 00000000000000000019.00000000000000000024.compacted.json : 19-24
   - 00000000000000000025.00000000000000000030.compacted.json : 25-30
   - 00000000000000000036.checkpoint.parquet
- So all of data from 1-6 jsons are dumped into compressed json file.
- For latest state calculation, compacted files are read.
- That is how it is optimised for performance.
- After 36 files have been created we see "checkpoint.parquet".
- It again condenses all of the information from 0 to 35th file and puts it over here.
- So if there is checkpoint file available , spark reads the latest checkpoint file and rest of JSON files thats it to produce the latest state of the table.
- So you dont end up wasting resources and end up reading all of normal of compacted json files, just the checkpoint latest and if any json files are present after that.
- So that is how Delta handles massive amount of JSON files.
- In above case : there 1 checkpoint and post that few compacted Jsons as shown below:
   - 00000000000000000036.checkpoint.parquet - all data is here till 36th json file.
   - 00000000000000000037.00000000000000000042.compacted.json
   - 00000000000000000043.00000000000000000048.compacted.json
   - 00000000000000000049.00000000000000000054.compacted.json
   - 00000000000000000049.json
   - 00000000000000000050.json
   - 00000000000000000051.json
   - 00000000000000000052.json
   - 00000000000000000053.json
   - 00000000000000000054.json
   - 00000000000000000055.json
   - _last_checkpoint
   - **_last_checkpoint** file is a pointer or a "shortcut."
   - Instead of Spark having to list thousands of files in the _delta_log folder to find the most recent state, it reads this tiny JSON file first. The _last_checkpoint file tells Spark:
   - The version number of the latest checkpoint (e.g., version 36).
   - The number of parts the checkpoint is split into (usually 1, unless the table is massive).
   - **Why it matters:** It provides an entry point so Spark can immediately jump to 000036.checkpoint.parquet rather than scanning the directory.
   - If you are at version 55 and have a checkpoint at 36, Spark reads 36.checkpoint.parquet + JSONs 37 through 55. It ignores all JSON files from 0 to 36.